# GS-Level Pipeline — Matrices Aggregated to GS Zones

Recreates the superzone-level products on the **GS zoning** (`Input/TAZ_GSnew.csv`, 25 zones covering all study TAZs) instead of `SZ_NEW`: weighted survey matrices, the cellular matrix, their comparison, the empirical-Bayes hybrid (with cross-day calibration of $k$), the hybrid trips matrix, and the GS-based correction factors carried down to the 778-TAZ matrices.

Methodology is identical to the superzone pipeline (`THS_2018_MTX_hybrid.ipynb`, `THS_2018_MTX_hybrid_taz.ipynb`) with the TAZ→GS mapping in place of TAZ→SZ_NEW.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, ORANGE = '#2a78d6', '#eb6834'
INK, INK2 = '#0b0b0b', '#52514e'

## 1. Inputs

In [2]:
df = pd.read_csv('Input/Matrices/ACTIVITIES_DEC18_corrected.csv')
weights = pd.read_csv('Input/Matrices/households_with_weights.csv')[['HHID', 'wf_new']]
df = df.merge(weights, on='HHID', how='left', validate='many_to_one')

gs_map = pd.read_csv('Input/TAZ_GSnew.csv')
taz_to_gs = gs_map.set_index('TAZ')['GS'].to_dict()
print(f"GS mapping: {gs_map['TAZ'].nunique()} TAZs -> {gs_map['GS'].nunique()} GS zones")

def am_peak_trips(df_day):
    d = df_day.sort_values(by=['INDIVID', 'tourID', 'ACT_ID'])
    d['origin'] = d.groupby(['INDIVID', 'tourID'])['taz'].shift(1)
    d['destination'] = d['taz']
    d['StartTime'] = pd.to_datetime(d['StartTime'], dayfirst=True)
    d['EndTime'] = pd.to_datetime(d['EndTime'], dayfirst=True)
    d['leaving_time'] = pd.to_datetime(np.where(d['mainActivity'] == 'Home', d['EndTime'], d['StartTime']))
    mask = (d['leaving_time'].dt.hour >= 6) & (d['leaving_time'].dt.hour < 9)
    return d[mask].dropna(subset=['origin', 'destination'])

t10 = am_peak_trips(df[df['ACT_DAY'] == 10])
t20 = am_peak_trips(df[df['ACT_DAY'] == 20])

# cellular at TAZ level (validated reconstruction)
cellular = pd.read_csv('Input/Matrices/AvgDayHourlyTrips201819_1270_weekday_v1.csv')[['fromZone', 'ToZone', 'h6', 'h7', 'h8']]
keys_raw = pd.read_csv('Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv', encoding='windows-1255')
keys = keys_raw[['TAZ_1270', 'TAZ_NUMBER']].dropna(subset=['TAZ_NUMBER'])
keys['TAZ_NUMBER'] = keys['TAZ_NUMBER'].astype(int)
keys = keys.drop_duplicates(subset=['TAZ_NUMBER'])
c = cellular.merge(keys.rename(columns={'TAZ_1270': 'fromZone', 'TAZ_NUMBER': 'fromTaz'}), on='fromZone')
c = c.merge(keys.rename(columns={'TAZ_1270': 'ToZone', 'TAZ_NUMBER': 'ToTaz'}), on='ToZone')
cell_taz = c.groupby(['fromTaz', 'ToTaz'])[['h6', 'h7', 'h8']].sum().sum(axis=1).unstack().fillna(0)
TAZ = cell_taz.index
P_cell_taz = cell_taz.div(cell_taz.sum(axis=1), axis=0).fillna(0)
assert all(t in taz_to_gs for t in TAZ), "GS mapping must cover all 778 TAZs"

GS = sorted(set(taz_to_gs[t] for t in TAZ))
gs_idx = {g: i for i, g in enumerate(GS)}
col_gs = np.array([gs_idx[taz_to_gs[t]] for t in TAZ])
M = np.zeros((len(TAZ), len(GS)))
M[np.arange(len(TAZ)), col_gs] = 1
print(f"GS zones present in the 778-TAZ system: {len(GS)}")

GS mapping: 781 TAZs -> 25 GS zones


GS zones present in the 778-TAZ system: 25


## 2. Survey and cellular matrices at GS level

In [3]:
def gs_survey(t, weighted=True):
    t = t.copy()
    t['origin_gs'] = t['origin'].map(taz_to_gs)
    t['dest_gs'] = t['destination'].map(taz_to_gs)
    t = t.dropna(subset=['origin_gs', 'dest_gs'])
    v = t.groupby(['origin_gs', 'dest_gs'])['wf_new'].sum() if weighted else t.groupby(['origin_gs', 'dest_gs']).size()
    return v.unstack().fillna(0).reindex(index=GS, columns=GS, fill_value=0)

W10_gs, W20_gs = gs_survey(t10, True), gs_survey(t20, True)
N10_gs, N20_gs = gs_survey(t10, False), gs_survey(t20, False)
n10_gs, n20_gs = N10_gs.sum(axis=1), N20_gs.sum(axis=1)
P10_gs = W10_gs.div(W10_gs.sum(axis=1), axis=0).fillna(0)
P20_gs = W20_gs.div(W20_gs.sum(axis=1), axis=0).fillna(0)

cell_gs = pd.DataFrame(M.T @ cell_taz.values @ M, index=GS, columns=GS)
P_cell_gs = cell_gs.div(cell_gs.sum(axis=1), axis=0).fillna(0)

print(f"expanded trips at GS level: day 10 = {W10_gs.sum().sum():,.0f}, day 20 = {W20_gs.sum().sum():,.0f}")
print(f"survey observations per origin GS: day 10 min {n10_gs.min():.0f}, median {n10_gs.median():.0f}, max {n10_gs.max():.0f}")

def rmse_corr(a, b):
    af, bf = a.values.flatten(), b.values.flatten()
    return np.sqrt(np.mean((af - bf) ** 2)), np.corrcoef(af, bf)[0, 1]

for label, p in [('Day 10 weighted', P10_gs), ('Day 20 weighted', P20_gs)]:
    r, cr = rmse_corr(p, P_cell_gs)
    print(f"{label} vs cellular at GS level:  RMSE {r:.4f},  Pearson r {cr:.4f}")

P10_gs.to_csv('Output/prob_gs_10_weighted.csv')
P20_gs.to_csv('Output/prob_gs_20_weighted.csv')
P_cell_gs.to_csv('Output/prob_gs_cellular.csv')
print("saved: prob_gs_10_weighted.csv, prob_gs_20_weighted.csv, prob_gs_cellular.csv")

expanded trips at GS level: day 10 = 2,193,510, day 20 = 2,163,320
survey observations per origin GS: day 10 min 5, median 373, max 4679
Day 10 weighted vs cellular at GS level:  RMSE 0.0892,  Pearson r 0.8399
Day 20 weighted vs cellular at GS level:  RMSE 0.0892,  Pearson r 0.8381
saved: prob_gs_10_weighted.csv, prob_gs_20_weighted.csv, prob_gs_cellular.csv


## 3. GS hybrid — empirical-Bayes shrinkage with cross-day calibration

In [4]:
def blend(P_s, n, k):
    lam = np.where(n.values > 0, n.values / (n.values + k), 0.0) if np.isfinite(k) else np.zeros(len(n))
    return pd.DataFrame(P_s.values * lam[:, None] + P_cell_gs.values * (1 - lam[:, None]), index=GS, columns=GS)

def mean_jsd(P_pred, P_t):
    vals = []
    for i in range(len(GS)):
        p, q = P_pred.values[i], P_t.values[i]
        if p.sum() <= 0 or q.sum() <= 0:
            continue
        p, q = p / p.sum(), q / q.sum()
        m = (p + q) / 2
        kl = lambda a, b: np.sum(a[a > 0] * np.log2(a[a > 0] / b[a > 0]))
        vals.append(0.5 * kl(p, m) + 0.5 * kl(q, m))
    return float(np.mean(vals))

ks = [0, 1, 2, 5, 10, 20, 50, 100, 200, 500, np.inf]
cv = pd.DataFrame([{'k': k,
                    'JSD 10->20': mean_jsd(blend(P10_gs, n10_gs, k), P20_gs),
                    'JSD 20->10': mean_jsd(blend(P20_gs, n20_gs, k), P10_gs)} for k in ks]).set_index('k')
cv['JSD avg'] = cv.mean(axis=1)
k_star = cv['JSD avg'].idxmin()
print(f"optimal k (JSD): {k_star:g}")
cv.round(5)

optimal k (JSD): 1


,JSD 10->20,JSD 20->10,JSD avg
k,,,
0.0,0.02517,0.02517,0.02517
1.0,0.02444,0.02514,0.02479
2.0,0.02461,0.02565,0.02513
5.0,0.02575,0.02727,0.02651
10.0,0.02754,0.02931,0.02843
20.0,0.03031,0.03220,0.03125
50.0,0.03678,0.03870,0.03774
100.0,0.04622,0.04812,0.04717
200.0,0.06194,0.06379,0.06286


In [5]:
W_pool_gs = W10_gs + W20_gs
P_pool_gs = W_pool_gs.div(W_pool_gs.sum(axis=1), axis=0).fillna(0)
n_pool_gs = n10_gs + n20_gs

K_STAR, K_SENS = (int(k_star) if np.isfinite(k_star) else 2), 100
def gs_hybrid(k):
    lam = (n_pool_gs / (n_pool_gs + k)).values if k > 0 else np.ones(len(GS))
    return pd.DataFrame(P_pool_gs.values * lam[:, None] + P_cell_gs.values * (1 - lam[:, None]), index=GS, columns=GS)

P_hybrid_gs = gs_hybrid(K_STAR) if K_STAR > 0 else gs_hybrid(1e-9)
P_hybrid_gs_k100 = gs_hybrid(K_SENS)

lam_tbl = pd.DataFrame({
    'n_A (pooled obs)': n_pool_gs.astype(int),
    f'lambda (k={K_STAR})': n_pool_gs / (n_pool_gs + K_STAR) if K_STAR > 0 else 1.0,
    f'lambda (k={K_SENS})': n_pool_gs / (n_pool_gs + K_SENS),
})
lam_tbl.index.name = 'origin_gs'

# trips: rows scaled to average-weekday expanded departures per origin GS
vol_gs = (W10_gs.sum(axis=1) + W20_gs.sum(axis=1)) / 2
T_hybrid_gs = P_hybrid_gs.mul(vol_gs, axis=0)

P_hybrid_gs.to_csv('Output/hybrid_gs_prob.csv')
P_hybrid_gs_k100.to_csv('Output/hybrid_gs_prob_k100.csv')
T_hybrid_gs.to_csv('Output/hybrid_gs_trips.csv')
lam_tbl.to_csv('Output/hybrid_gs_lambda.csv')
cv.to_csv('Output/hybrid_gs_cv_results.csv')
print(f"k* = {K_STAR} | lambda range: {lam_tbl.iloc[:, 1].min():.4f}-{lam_tbl.iloc[:, 1].max():.4f}")
print(f"hybrid GS trips total (avg weekday AM peak): {T_hybrid_gs.sum().sum():,.0f}")
print("saved: hybrid_gs_prob.csv, hybrid_gs_prob_k100.csv, hybrid_gs_trips.csv, hybrid_gs_lambda.csv, hybrid_gs_cv_results.csv")

k* = 1 | lambda range: 0.9091-0.9999
hybrid GS trips total (avg weekday AM peak): 2,178,415
saved: hybrid_gs_prob.csv, hybrid_gs_prob_k100.csv, hybrid_gs_trips.csv, hybrid_gs_lambda.csv, hybrid_gs_cv_results.csv


## 4. GS correction factors → TAZ-level matrices

Same construction as the superzone-based step 3, with GS in place of `SZ_NEW`:
$R_{AB} = P^*(B|A)/P^{cell}(B|A)$ per GS pair, $\tilde C_{ij} = C_{ij} R_{AB}$, rows normalized; trips scaled by GS survey departure volumes split within GS by cellular outflow shares.

In [6]:
R_gs = np.where(P_cell_gs.values > 0,
                P_hybrid_gs.values / np.where(P_cell_gs.values == 0, 1, P_cell_gs.values), 1.0)
C_tilde = cell_taz.values * R_gs[col_gs][:, col_gs]
rs = C_tilde.sum(axis=1, keepdims=True)
P_taz_hybrid_gs = pd.DataFrame(np.where(rs > 0, C_tilde / rs, 0), index=TAZ, columns=TAZ)

diag = np.diag(R_gs)
off = R_gs[~np.eye(len(GS), dtype=bool)]
zero_pairs = ((N10_gs + N20_gs).values[~np.eye(len(GS), dtype=bool)] == 0).mean()
print(f"R_AB: diagonal median {np.median(diag):.2f} (max {diag.max():.1f}) | "
      f"off-diagonal median {np.median(off):.3f} | GS pairs with zero pooled survey obs: {zero_pairs:.0%}")

# origin volumes: GS survey departures split by cellular outflow shares
cell_out = cell_taz.sum(axis=1).values
share = np.zeros(len(TAZ))
for g_i in range(len(GS)):
    m = col_gs == g_i
    tot = cell_out[m].sum()
    share[m] = cell_out[m] / tot if tot > 0 else 1.0 / m.sum()
T_origin = vol_gs.values[col_gs] * share
T_taz_hybrid_gs = P_taz_hybrid_gs.mul(T_origin, axis=0)

gs_check = pd.Series(M.T @ T_taz_hybrid_gs.sum(axis=1).values, index=GS)
assert np.allclose(gs_check, vol_gs)
print(f"TAZ trips total: {T_taz_hybrid_gs.sum().sum():,.0f} — GS origin totals match survey departures")

pd.DataFrame(R_gs, index=GS, columns=GS).to_csv('Output/gs_correction_factors.csv', float_format='%.6g')
P_taz_hybrid_gs.to_csv('Output/hybrid_taz_prob_gs.csv', float_format='%.6g')
T_taz_hybrid_gs.to_csv('Output/hybrid_taz_trips_gs.csv', float_format='%.6g')
print("saved: gs_correction_factors.csv, hybrid_taz_prob_gs.csv, hybrid_taz_trips_gs.csv")

R_AB: diagonal median 1.89 (max 5.7) | off-diagonal median 0.091 | GS pairs with zero pooled survey obs: 48%
TAZ trips total: 2,178,415 — GS origin totals match survey departures


saved: gs_correction_factors.csv, hybrid_taz_prob_gs.csv, hybrid_taz_trips_gs.csv


## Notes

- The GS mapping covers every TAZ in the 778-zone system (781 mapped TAZs, including 105 which the keys table lacks an `SZ_NEW` for; the survey-side aggregation uses the full mapping, so trips touching TAZ 105 are included at GS level).
- With only 25 origin zones the survey sample per origin is even richer than at superzone level, so — as at superzone level — cross-day validation selects (near-)zero shrinkage and the hybrid is survey-dominant with a light cellular floor; the `k=100` variant keeps a larger cellular floor on survey-unobserved GS pairs.
- All superzone-level caveats carry over unchanged: the cellular replication mapping, the intra-zone survey/cellular divergence, and the same-households limitation of cross-day validation.